In [0]:
products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("/Volumes/ecommerce/bronze/raw_files/products.csv")
)

display(products_df)

product_id,product_name,category,subcategory,unit_price,stock_quantity
P001,Wireless Mouse,Electronics,Accessories,799,100
P002,Mechanical Keyboard,Electronics,Accessories,2499,75
P003,USB-C Hub,Electronics,Accessories,1499,120
P004,Laptop Stand,Electronics,Office,1899,60
P005,Water Bottle,Home & Living,Kitchen,699,200
P006,Backpack,Fashion,Bags,1599,90
P007,Running Shoes,Fashion,Footwear,3299,50
P008,Cotton T-Shirt,Fashion,Clothing,899,150
P009,Desk Lamp,Home & Living,Office,1299,80
P010,Bluetooth Speaker,Electronics,Audio,2199,65


In [0]:
%sql
SELECT
    order_date,
    COUNT(order_id) AS total_orders,
    SUM(order_total) AS total_sales
FROM ecommerce.silver.orders
GROUP BY order_date
ORDER BY order_date;

order_date,total_orders,total_sales
2026-09-01,1,2298
2026-09-02,1,2499
2026-09-03,1,1599
2026-09-04,1,699
2026-09-05,1,3299
2026-09-06,1,2199
2026-09-07,1,1899
2026-09-08,1,899
2026-09-09,1,3997
2026-09-10,1,1499


In [0]:
%sql
SELECT
    c.customer_id,
    c.customer_name,
    c.email,
    c.country,
    c.state,
    COUNT(o.order_id) AS total_orders,
    COALESCE(SUM(o.order_total), 0) AS total_spent
FROM ecommerce.silver.customers c
LEFT JOIN ecommerce.silver.orders o
    ON c.customer_id = o.customer_id
GROUP BY
    c.customer_id,
    c.customer_name,
    c.email,
    c.country,
    c.state
ORDER BY total_spent DESC;

customer_id,customer_name,email,country,state,total_orders,total_spent
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2,6897
C009,Nisha George,nisha.george@example.com,India,Kerala,1,3997
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2,3299
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,1,2298
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,1,2199
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,1,1899
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,1,1599
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,1,1499
C008,Kiran Das,kiran.das@example.com,India,West Bengal,1,899
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,1,699


In [0]:
%sql
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.subcategory,
    SUM(oi.quantity) AS units_sold,
    SUM(oi.line_amount) AS total_revenue
FROM ecommerce.silver.order_items oi
INNER JOIN ecommerce.silver.products p
    ON oi.product_id = p.product_id
INNER JOIN ecommerce.silver.orders o
    ON oi.order_id = o.order_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category,
    p.subcategory
ORDER BY total_revenue DESC;

product_id,product_name,category,subcategory,units_sold,total_revenue
P007,Running Shoes,Fashion,Footwear,2,6598.0
P010,Bluetooth Speaker,Electronics,Audio,3,6597.0
P002,Mechanical Keyboard,Electronics,Accessories,1,2499.0
P004,Laptop Stand,Electronics,Office,1,1899.0
P008,Cotton T-Shirt,Fashion,Clothing,2,1798.0
P006,Backpack,Fashion,Bags,1,1599.0
P001,Wireless Mouse,Electronics,Accessories,2,1598.0
P003,USB-C Hub,Electronics,Accessories,1,1499.0
P005,Water Bottle,Home & Living,Kitchen,2,1398.0


In [0]:
%sql
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.subcategory,
    SUM(oi.quantity) AS units_sold,
    SUM(oi.line_amount) AS total_revenue
FROM ecommerce.silver.order_items oi
INNER JOIN ecommerce.silver.products p
    ON oi.product_id = p.product_id
INNER JOIN ecommerce.silver.orders o
    ON oi.order_id = o.order_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category,
    p.subcategory
ORDER BY total_revenue DESC;

product_id,product_name,category,subcategory,units_sold,total_revenue
P007,Running Shoes,Fashion,Footwear,2,6598.0
P010,Bluetooth Speaker,Electronics,Audio,3,6597.0
P002,Mechanical Keyboard,Electronics,Accessories,1,2499.0
P004,Laptop Stand,Electronics,Office,1,1899.0
P008,Cotton T-Shirt,Fashion,Clothing,2,1798.0
P006,Backpack,Fashion,Bags,1,1599.0
P001,Wireless Mouse,Electronics,Accessories,2,1598.0
P003,USB-C Hub,Electronics,Accessories,1,1499.0
P005,Water Bottle,Home & Living,Kitchen,2,1398.0


In [0]:
%sql
SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.order_date,
    o.order_status,
    o.order_total,
    p.payment_method,
    p.payment_status,
    p.payment_date
FROM ecommerce.silver.orders o
LEFT JOIN ecommerce.silver.customers c
    ON o.customer_id = c.customer_id
LEFT JOIN ecommerce.silver.payments p
    ON o.order_id = p.order_id
ORDER BY o.order_date;

order_id,customer_id,customer_name,order_date,order_status,order_total,payment_method,payment_status,payment_date
O1001,C001,Ananya Nair,2026-09-01,Delivered,2298,UPI,Paid,2026-09-01
O1002,C002,Rahul Menon,2026-09-02,Shipped,2499,Credit Card,Paid,2026-09-02
O1003,C003,Meera Thomas,2026-09-03,Delivered,1599,Debit Card,Paid,2026-09-03
O1004,C004,Arjun Kumar,2026-09-04,Cancelled,699,UPI,Refunded,2026-09-04
O1005,C005,Diya Joseph,2026-09-05,Delivered,3299,Credit Card,Paid,2026-09-05
O1006,C006,Vivek Shah,2026-09-06,Processing,2199,UPI,Pending,2026-09-06
O1007,C007,Sneha Iyer,2026-09-07,Delivered,1899,Net Banking,Paid,2026-09-07
O1008,C008,Kiran Das,2026-09-08,Returned,899,UPI,Refunded,2026-09-08
O1009,C009,Nisha George,2026-09-09,Delivered,3997,Credit Card,Paid,2026-09-09
O1010,C010,Aditya Rao,2026-09-10,Shipped,1499,UPI,Paid,2026-09-10


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.products
USING DELTA
LOCATION 'abfss://silver@ecommerce1darsana.dfs.core.windows.net/products';

In [0]:
from pyspark.sql.functions import col, sum

products_clean_df = (
    products_df
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("stock_quantity", col("stock_quantity").cast("int"))
)

print("Products Bronze rows:", products_df.count())

print("\nNull values after type conversion:")
products_clean_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in products_clean_df.columns
    ]
).show()

print("\nDuplicate product IDs:",
      products_clean_df
      .groupBy("product_id")
      .count()
      .filter("count > 1")
      .count())

display(products_clean_df)

Products Bronze rows: 10

Null values after type conversion:
+----------+------------+--------+-----------+----------+--------------+
|product_id|product_name|category|subcategory|unit_price|stock_quantity|
+----------+------------+--------+-----------+----------+--------------+
|         0|           0|       0|          0|         0|             0|
+----------+------------+--------+-----------+----------+--------------+


Duplicate product IDs: 0


product_id,product_name,category,subcategory,unit_price,stock_quantity
P001,Wireless Mouse,Electronics,Accessories,799.0,100
P002,Mechanical Keyboard,Electronics,Accessories,2499.0,75
P003,USB-C Hub,Electronics,Accessories,1499.0,120
P004,Laptop Stand,Electronics,Office,1899.0,60
P005,Water Bottle,Home & Living,Kitchen,699.0,200
P006,Backpack,Fashion,Bags,1599.0,90
P007,Running Shoes,Fashion,Footwear,3299.0,50
P008,Cotton T-Shirt,Fashion,Clothing,899.0,150
P009,Desk Lamp,Home & Living,Office,1299.0,80
P010,Bluetooth Speaker,Electronics,Audio,2199.0,65


In [0]:
products_silver_path = \
    "abfss://silver@ecommerce1darsana.dfs.core.windows.net/products"

products_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(products_silver_path)

print("Products Silver written successfully.")

Products Silver written successfully.


In [0]:

# Read Products from Bronze
products_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("/Volumes/ecommerce/bronze/raw_files/products.csv")

print("Products Bronze rows:", products_df.count())
products_df.printSchema()
display(products_df)

Products Bronze rows: 10
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- stock_quantity: string (nullable = true)



product_id,product_name,category,subcategory,unit_price,stock_quantity
P001,Wireless Mouse,Electronics,Accessories,799,100
P002,Mechanical Keyboard,Electronics,Accessories,2499,75
P003,USB-C Hub,Electronics,Accessories,1499,120
P004,Laptop Stand,Electronics,Office,1899,60
P005,Water Bottle,Home & Living,Kitchen,699,200
P006,Backpack,Fashion,Bags,1599,90
P007,Running Shoes,Fashion,Footwear,3299,50
P008,Cotton T-Shirt,Fashion,Clothing,899,150
P009,Desk Lamp,Home & Living,Office,1299,80
P010,Bluetooth Speaker,Electronics,Audio,2199,65


In [0]:
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/ecommerce/bronze/raw_files/orders.csv")

In [0]:
duplicate_order_count = (
    orders_df
    .groupBy("order_id")
    .count()
    .filter("count > 1")
    .count()
)

print("Duplicate order IDs:", duplicate_order_count)

Duplicate order IDs: 1


In [0]:
orders_df.groupBy("order_id") \
    .count() \
    .filter("count > 1") \
    .show()

+--------+-----+
|order_id|count|
+--------+-----+
|   O1012|    2|
+--------+-----+



In [0]:
display(
    orders_df.filter(orders_df.order_id == "O1012")
)

order_id,customer_id,order_date,order_status,order_total
O1012,C002,2026-09-12,Delivered,4398
O1012,C002,2026-09-12,Delivered,4398


In [0]:
from pyspark.sql.functions import col, coalesce, expr, to_date

# Remove duplicate order IDs
orders_clean_df = orders_df.dropDuplicates(["order_id"])

# Safely parse both supported date formats
orders_clean_df = orders_clean_df.withColumn(
    "order_date",
    to_date(
        coalesce(
            expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
            expr("try_to_timestamp(order_date, 'yyyy/MM/dd')")
        )
    )
)

orders_clean_df.printSchema()
display(orders_clean_df)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



order_id,customer_id,order_date,order_status,order_total
O1001,C001,2026-09-01,Delivered,2298
O1002,C002,2026-09-02,Shipped,2499
O1003,C003,2026-09-03,Delivered,1599
O1004,C004,2026-09-04,Cancelled,699
O1005,C005,2026-09-05,Delivered,3299
O1006,C006,2026-09-06,Processing,2199
O1007,C007,2026-09-07,Delivered,1899
O1008,C008,2026-09-08,Returned,899
O1009,C009,2026-09-09,Delivered,3997
O1010,C010,2026-09-10,Shipped,1499


In [0]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



In [0]:
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/ecommerce/bronze/raw_files/orders.csv")

orders_clean_df = orders_df.dropDuplicates(["order_id"])

print("Bronze rows:", orders_df.count())
print("Silver-ready rows:", orders_clean_df.count())

Bronze rows: 14
Silver-ready rows: 13


In [0]:
from pyspark.sql.functions import col, coalesce, expr, to_date

# 1. Read raw Bronze data
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("/Volumes/ecommerce/bronze/raw_files/orders.csv")

# 2. Remove duplicate Order IDs
orders_clean_df = orders_df.dropDuplicates(["order_id"])

# 3. Parse both date formats safely
orders_clean_df = orders_clean_df.withColumn(
    "order_date",
    to_date(
        coalesce(
            expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
            expr("try_to_timestamp(order_date, 'yyyy/MM/dd')")
        )
    )
)

# 4. Convert order_total to integer
orders_clean_df = orders_clean_df.withColumn(
    "order_total",
    expr("try_cast(order_total AS INT)")
)

# 5. Materialize and verify
orders_clean_df = orders_clean_df.cache()
orders_clean_df.count()

orders_clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col, to_date, coalesce

# Read Bronze again
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("/Volumes/ecommerce/bronze/raw_files/orders.csv")

# Remove exact duplicate order IDs
orders_clean_df = orders_df.dropDuplicates(["order_id"])

# Convert both possible date formats safely
orders_clean_df = orders_clean_df.withColumn(
    "order_date",
    coalesce(
        to_date(col("order_date"), "yyyy-MM-dd"),
        to_date(col("order_date"), "yyyy/MM/dd")
    )
)

orders_clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, to_date, coalesce

# 1. Read raw Bronze data
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("/Volumes/ecommerce/bronze/raw_files/orders.csv")

# 2. Remove duplicate Order IDs
orders_clean_df = orders_df.dropDuplicates(["order_id"])

# 3. Convert order_date safely
from pyspark.sql.functions import expr, coalesce, to_date

orders_clean_df = orders_clean_df.withColumn(
    "order_date",
    to_date(
        coalesce(
            expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
            expr("try_to_timestamp(order_date, 'yyyy/MM/dd')")
        )
    )
)

# 4. Convert order_total to integer
orders_clean_df = orders_clean_df.withColumn(
    "order_total",
    col("order_total").cast("int")
)

# 5. Materialize the DataFrame
orders_clean_df = orders_clean_df.cache()
orders_clean_df.count()

orders_clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col, try_to_date, coalesce

# Read Bronze as strings
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("/Volumes/ecommerce/bronze/raw_files/orders.csv")

# Remove duplicate order IDs
orders_clean_df = orders_df.dropDuplicates(["order_id"])

# Safely handle both date formats
orders_clean_df = orders_clean_df.withColumn(
    "order_date",
    coalesce(
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "yyyy/MM/dd")
    )
)

# Convert order total
orders_clean_df = orders_clean_df.withColumn(
    "order_total",
    col("order_total").cast("int")
)

orders_clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



In [0]:
valid_customer_ids = (
    spark.table("ecommerce.silver.customers")
    .select("customer_id")
)

orders_silver_df = (
    orders_clean_df
    .join(
        valid_customer_ids,
        "customer_id",
        "inner"
    )
)

print("Orders before referential check:", orders_clean_df.count())
print("Valid Silver orders:", orders_silver_df.count())

Orders before referential check: 13
Valid Silver orders: 12


In [0]:
orders_quarantine_df = (
    orders_clean_df
    .join(
        valid_customer_ids,
        "customer_id",
        "left_anti"
    )
)

display(orders_quarantine_df)

customer_id,order_id,order_date,order_status,order_total
C999,O1011,2026-09-11,Processing,799


In [0]:

from pyspark.sql.functions import col, coalesce, to_date

orders_quarantine_df = orders_quarantine_df.withColumn(
    "order_date",
    coalesce(
        to_date(col("order_date"), "yyyy-MM-dd"),
        to_date(col("order_date"), "yyyy/MM/dd")
    )
)

orders_quarantine_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



In [0]:
products_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("/Volumes/ecommerce/bronze/raw_files/products.csv")

payments_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("/Volumes/ecommerce/bronze/raw_files/payments.csv")

order_items_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("/Volumes/ecommerce/bronze/raw_files/order_items.csv")

In [0]:
from pyspark.sql.functions import col, sum

datasets = {
    "PRODUCTS": products_df,
    "PAYMENTS": payments_df,
    "ORDER_ITEMS": order_items_df
}

for name, df in datasets.items():

    print(f"\n{'='*50}")
    print(f"{name} DATA QUALITY CHECK")
    print(f"{'='*50}")

    # Row count
    print("Rows:", df.count())

    # Null values
    print("\nNull values:")
    df.select(
        *[
            sum(col(c).isNull().cast("int")).alias(c)
            for c in df.columns
        ]
    ).show()

    # Duplicate primary IDs
    id_column = {
        "PRODUCTS": "product_id",
        "PAYMENTS": "payment_id",
        "ORDER_ITEMS": "order_item_id"
    }[name]

    duplicate_count = (
        df.groupBy(id_column)
          .count()
          .filter("count > 1")
          .count()
    )

    print(f"Duplicate {id_column}s:", duplicate_count)


PRODUCTS DATA QUALITY CHECK
Rows: 10

Null values:
+----------+------------+--------+-----------+----------+--------------+
|product_id|product_name|category|subcategory|unit_price|stock_quantity|
+----------+------------+--------+-----------+----------+--------------+
|         0|           0|       0|          0|         0|             0|
+----------+------------+--------+-----------+----------+--------------+

Duplicate product_ids: 0

PAYMENTS DATA QUALITY CHECK
Rows: 13

Null values:
+----------+--------+--------------+------+--------------+------------+
|payment_id|order_id|payment_method|amount|payment_status|payment_date|
+----------+--------+--------------+------+--------------+------------+
|         0|       0|             0|     0|             0|           0|
+----------+--------+--------------+------+--------------+------------+

Duplicate payment_ids: 0

ORDER_ITEMS DATA QUALITY CHECK
Rows: 16

Null values:
+-------------+--------+----------+--------+-----------+
|order_

In [0]:
from pyspark.sql.functions import expr, col

print("========== PRODUCTS ==========")

print("Rows:", products_df.count())

print("Invalid unit prices:",
      products_df.filter(
          expr("try_cast(unit_price AS DOUBLE) IS NULL")
      ).count())

print("Invalid stock quantities:",
      products_df.filter(
          expr("try_cast(stock_quantity AS INT) IS NULL")
      ).count())

print("Negative unit prices:",
      products_df.filter(
          expr("try_cast(unit_price AS DOUBLE) < 0")
      ).count())

print("Negative stock quantities:",
      products_df.filter(
          expr("try_cast(stock_quantity AS INT) < 0")
      ).count())


print("\n========== PAYMENTS ==========")

print("Rows:", payments_df.count())

print("Invalid payment amounts:",
      payments_df.filter(
          expr("try_cast(amount AS DOUBLE) IS NULL")
      ).count())

print("Negative payment amounts:",
      payments_df.filter(
          expr("try_cast(amount AS DOUBLE) < 0")
      ).count())

print("Invalid payment dates:",
      payments_df.filter(
          expr("""
              coalesce(
                  try_to_timestamp(payment_date, 'yyyy-MM-dd'),
                  try_to_timestamp(payment_date, 'yyyy/MM/dd')
              ) IS NULL
          """)
      ).count())

print("\nPayment statuses:")
payments_df.groupBy("payment_status").count().show()


print("\n========== ORDER ITEMS ==========")

print("Rows:", order_items_df.count())

print("Invalid quantities:",
      order_items_df.filter(
          expr("try_cast(quantity AS INT) IS NULL")
      ).count())

print("Invalid line amounts:",
      order_items_df.filter(
          expr("try_cast(line_amount AS DOUBLE) IS NULL")
      ).count())

print("Negative quantities:",
      order_items_df.filter(
          expr("try_cast(quantity AS INT) < 0")
      ).count())

print("Negative line amounts:",
      order_items_df.filter(
          expr("try_cast(line_amount AS DOUBLE) < 0")
      ).count())

========== PRODUCTS ==========
Rows: 10
Invalid unit prices: 0
Invalid stock quantities: 0
Negative unit prices: 0
Negative stock quantities: 0

========== PAYMENTS ==========
Rows: 13
Invalid payment amounts: 0
Negative payment amounts: 0
Invalid payment dates: 0

Payment statuses:
+--------------+-----+
|payment_status|count|
+--------------+-----+
|      Refunded|    2|
|        Failed|    2|
|       Pending|    1|
|          Paid|    8|
+--------------+-----+


========== ORDER ITEMS ==========
Rows: 16
Invalid quantities: 0
Invalid line amounts: 0
Negative quantities: 0
Negative line amounts: 0


In [0]:
from pyspark.sql.functions import col, expr, try_to_date

print("========== PRODUCTS ==========")

print("Rows:", products_df.count())

print("Invalid unit prices:",
      products_df.filter(
          expr("try_cast(unit_price AS DOUBLE) IS NULL")
      ).count())

print("Invalid stock quantities:",
      products_df.filter(
          expr("try_cast(stock_quantity AS INT) IS NULL")
      ).count())

print("Negative unit prices:",
      products_df.filter(
          expr("try_cast(unit_price AS DOUBLE) < 0")
      ).count())

print("Negative stock quantities:",
      products_df.filter(
          expr("try_cast(stock_quantity AS INT) < 0")
      ).count())


print("\n========== PAYMENTS ==========")

print("Rows:", payments_df.count())

print("Invalid payment amounts:",
      payments_df.filter(
          expr("try_cast(amount AS DOUBLE) IS NULL")
      ).count())

print("Negative payment amounts:",
      payments_df.filter(
          expr("try_cast(amount AS DOUBLE) < 0")
      ).count())

print("Invalid payment dates:",
      payments_df.filter(
          try_to_date(col("payment_date"), "yyyy-MM-dd").isNull()
      ).count())

print("\nPayment statuses:")
payments_df.groupBy("payment_status").count().show()


print("\n========== ORDER ITEMS ==========")

print("Rows:", order_items_df.count())

print("Invalid quantities:",
      order_items_df.filter(
          expr("try_cast(quantity AS INT) IS NULL")
      ).count())

print("Invalid line amounts:",
      order_items_df.filter(
          expr("try_cast(line_amount AS DOUBLE) IS NULL")
      ).count())

print("Negative quantities:",
      order_items_df.filter(
          expr("try_cast(quantity AS INT) < 0")
      ).count())

print("Negative line amounts:",
      order_items_df.filter(
          expr("try_cast(line_amount AS DOUBLE) < 0")
      ).count())

========== PRODUCTS ==========
Rows: 10
Invalid unit prices: 0
Invalid stock quantities: 0
Negative unit prices: 0
Negative stock quantities: 0

========== PAYMENTS ==========
Rows: 13
Invalid payment amounts: 0
Negative payment amounts: 0
Invalid payment dates: 0

Payment statuses:
+--------------+-----+
|payment_status|count|
+--------------+-----+
|      Refunded|    2|
|        Failed|    2|
|       Pending|    1|
|          Paid|    8|
+--------------+-----+


========== ORDER ITEMS ==========
Rows: 16
Invalid quantities: 0
Invalid line amounts: 0
Negative quantities: 0
Negative line amounts: 0


In [0]:
print("========== REFERENTIAL INTEGRITY ==========")

# Orders Silver
orders_ids = (
    spark.table("ecommerce.silver.orders")
    .select("order_id")
    .distinct()
)

# Products Bronze
product_ids = (
    products_df
    .select("product_id")
    .distinct()
)

# Payments → Orders
invalid_payment_orders = (
    payments_df
    .join(orders_ids, "order_id", "left_anti")
    .count()
)

print("Payments with invalid order IDs:", invalid_payment_orders)


# Order Items → Orders
invalid_item_orders = (
    order_items_df
    .join(orders_ids, "order_id", "left_anti")
    .count()
)

print("Order Items with invalid order IDs:", invalid_item_orders)


# Order Items → Products
invalid_item_products = (
    order_items_df
    .join(product_ids, "product_id", "left_anti")
    .count()
)

print("Order Items with invalid product IDs:", invalid_item_products)

========== REFERENTIAL INTEGRITY ==========
Payments with invalid order IDs: 1
Order Items with invalid order IDs: 1
Order Items with invalid product IDs: 0


In [0]:
print("========== INVALID PAYMENT ==========")

invalid_payment_df = (
    payments_df
    .join(orders_ids, "order_id", "left_anti")
)

display(invalid_payment_df)


print("========== INVALID ORDER ITEM ==========")

invalid_order_item_df = (
    order_items_df
    .join(orders_ids, "order_id", "left_anti")
)

display(invalid_order_item_df)

========== INVALID PAYMENT ==========


order_id,payment_id,payment_method,amount,payment_status,payment_date
O1011,PAY011,UPI,799,Failed,2026-09-11


========== INVALID ORDER ITEM ==========


order_id,order_item_id,product_id,quantity,line_amount
O1011,OI014,P001,1,799


In [0]:
payments_silver_df = (
    payments_df
    .join(
        orders_ids,
        "order_id",
        "inner"
    )
)

payments_silver_df = payments_silver_df.withColumn(
    "amount",
    col("amount").cast("double")
)

payments_silver_df = payments_silver_df.withColumn(
    "payment_date",
    try_to_date(col("payment_date"), "yyyy-MM-dd")
)

print("Payments Bronze rows:", payments_df.count())
print("Valid Payments Silver rows:", payments_silver_df.count())

display(payments_silver_df)

Payments Bronze rows: 13
Valid Payments Silver rows: 12


order_id,payment_id,payment_method,amount,payment_status,payment_date
O1006,PAY006,UPI,2199.0,Pending,2026-09-06
O1002,PAY002,Credit Card,2499.0,Paid,2026-09-02
O1004,PAY004,UPI,699.0,Refunded,2026-09-04
O1009,PAY009,Credit Card,3997.0,Paid,2026-09-09
O1007,PAY007,Net Banking,1899.0,Paid,2026-09-07
O1001,PAY001,UPI,2298.0,Paid,2026-09-01
O1012,PAY012,Credit Card,4398.0,Paid,2026-09-12
O1003,PAY003,Debit Card,1599.0,Paid,2026-09-03
O1008,PAY008,UPI,899.0,Refunded,2026-09-08
O1013,PAY013,UPI,0.0,Failed,2026-09-13


In [0]:
from pyspark.sql.functions import col, coalesce, expr, to_date

# Convert payment columns to their correct data types
payments_clean_df = (
    payments_df
    .withColumn("amount", expr("try_cast(amount AS DOUBLE)"))
    .withColumn(
        "payment_date",
        to_date(
            coalesce(
                expr("try_to_timestamp(payment_date, 'yyyy-MM-dd')"),
                expr("try_to_timestamp(payment_date, 'yyyy/MM/dd')")
            )
        )
    )
)

# Keep only payments linked to valid orders
payments_clean_df = payments_clean_df.join(
    orders_ids,
    "order_id",
    "inner"
)

print("Cleaned payments:", payments_clean_df.count())
payments_clean_df.printSchema()
display(payments_clean_df)

Cleaned payments: 12
root
 |-- order_id: string (nullable = true)
 |-- payment_id: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_date: date (nullable = true)



order_id,payment_id,payment_method,amount,payment_status,payment_date
O1006,PAY006,UPI,2199.0,Pending,2026-09-06
O1002,PAY002,Credit Card,2499.0,Paid,2026-09-02
O1004,PAY004,UPI,699.0,Refunded,2026-09-04
O1009,PAY009,Credit Card,3997.0,Paid,2026-09-09
O1007,PAY007,Net Banking,1899.0,Paid,2026-09-07
O1001,PAY001,UPI,2298.0,Paid,2026-09-01
O1012,PAY012,Credit Card,4398.0,Paid,2026-09-12
O1003,PAY003,Debit Card,1599.0,Paid,2026-09-03
O1008,PAY008,UPI,899.0,Refunded,2026-09-08
O1013,PAY013,UPI,0.0,Failed,2026-09-13


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.payments
USING DELTA
LOCATION 'abfss://silver@ecommerce1darsana.dfs.core.windows.net/payments';

CREATE TABLE IF NOT EXISTS ecommerce.silver.order_items
USING DELTA
LOCATION 'abfss://silver@ecommerce1darsana.dfs.core.windows.net/order_items';

In [0]:
order_items_silver_df = (
    order_items_df
    .join(
        orders_ids,
        "order_id",
        "inner"
    )
)

order_items_silver_df = order_items_silver_df.withColumn(
    "quantity",
    col("quantity").cast("int")
)

order_items_silver_df = order_items_silver_df.withColumn(
    "line_amount",
    col("line_amount").cast("double")
)

print("Order Items Bronze rows:", order_items_df.count())
print("Valid Order Items Silver rows:", order_items_silver_df.count())

display(order_items_silver_df)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6225517185963875>, line 2
      1 order_items_silver_df = (
----> 2     order_items_df
      3     .join(
      4         orders_ids,
      5         "order_id",
      6         "inner"
      7     )
      8 )
     10 order_items_silver_df = order_items_silver_df.withColumn(
     11     "quantity",
     12     col("quantity").cast("int")
     13 )
     15 order_items_silver_df = order_items_silver_df.withColumn(
     16     "line_amount",
     17     col("line_amount").cast("double")
     18 )

NameError: name 'order_items_df' is not defined

In [0]:
order_items_silver_path = \
    "abfss://silver@ecommerce1darsana.dfs.core.windows.net/order_items"

order_items_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(order_items_silver_path)

print("Order Items Silver written successfully.")

Order Items Silver written successfully.


In [0]:
from pyspark.sql.functions import col, coalesce, to_date

# Invalid Orders
orders_quarantine_df = (
    orders_clean_df
    .join(valid_customer_ids, "customer_id", "left_anti")
    .withColumn(
        "order_date",
        coalesce(
            to_date(col("order_date"), "yyyy-MM-dd"),
            to_date(col("order_date"), "yyyy/MM/dd")
        )
    )
)

# Invalid Payments
payments_quarantine_df = (
    payments_df
    .join(orders_ids, "order_id", "left_anti")
)

# Invalid Order Items
order_items_quarantine_df = (
    order_items_df
    .join(orders_ids, "order_id", "left_anti")
)

print("Quarantined Orders:", orders_quarantine_df.count())
print("Quarantined Payments:", payments_quarantine_df.count())
print("Quarantined Order Items:", order_items_quarantine_df.count())

orders_quarantine_df.printSchema()

display(orders_quarantine_df)
display(payments_quarantine_df)
display(order_items_quarantine_df)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6225517185963876>, line 5
      1 from pyspark.sql.functions import col, coalesce, to_date
      3 # Invalid Orders
      4 orders_quarantine_df = (
----> 5     orders_clean_df
      6     .join(valid_customer_ids, "customer_id", "left_anti")
      7     .withColumn(
      8         "order_date",
      9         coalesce(
     10             to_date(col("order_date"), "yyyy-MM-dd"),
     11             to_date(col("order_date"), "yyyy/MM/dd")
     12         )
     13     )
     14 )
     16 # Invalid Payments
     17 payments_quarantine_df = (
     18     payments_df
     19     .join(orders_ids, "order_id", "left_anti")
     20 )

NameError: name 'orders_clean_df' is not defined

In [0]:
orders_quarantine_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@ecommerce1darsana.dfs.core.windows.net/quarantine/orders")

payments_quarantine_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@ecommerce1darsana.dfs.core.windows.net/quarantine/payments")

order_items_quarantine_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@ecommerce1darsana.dfs.core.windows.net/quarantine/order_items")

print("All quarantine records saved successfully.")

All quarantine records saved successfully.


In [0]:

from pyspark.sql.functions import expr

orders_quarantine_df = orders_quarantine_df.withColumn(
    "order_date",
    expr("""
        coalesce(
            try_cast(try_to_timestamp(order_date, 'yyyy-MM-dd') AS DATE),
            try_cast(try_to_timestamp(order_date, 'yyyy/MM/dd') AS DATE)
        )
    """)
)

orders_quarantine_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



In [0]:
payments_silver_path = (
    "abfss://silver@ecommerce1darsana.dfs.core.windows.net/payments"
)

payments_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(payments_silver_path)

print("Payments Silver written successfully.")

Payments Silver written successfully.


In [0]:
display(invalid_order_item_df)

order_id,order_item_id,product_id,quantity,line_amount
O1011,OI014,P001,1,799


In [0]:
products_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("/Volumes/ecommerce/bronze/raw_files/products.csv")

display(products_df)

products_df.printSchema()

product_id,product_name,category,subcategory,unit_price,stock_quantity
P001,Wireless Mouse,Electronics,Accessories,799,100
P002,Mechanical Keyboard,Electronics,Accessories,2499,75
P003,USB-C Hub,Electronics,Accessories,1499,120
P004,Laptop Stand,Electronics,Office,1899,60
P005,Water Bottle,Home & Living,Kitchen,699,200
P006,Backpack,Fashion,Bags,1599,90
P007,Running Shoes,Fashion,Footwear,3299,50
P008,Cotton T-Shirt,Fashion,Clothing,899,150
P009,Desk Lamp,Home & Living,Office,1299,80
P010,Bluetooth Speaker,Electronics,Audio,2199,65


root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- stock_quantity: string (nullable = true)



In [0]:
%sql
SELECT * FROM ecommerce.silver.orders;

customer_id,order_id,order_date,order_status,order_total
C001,O1001,2026-09-01,Delivered,2298
C002,O1002,2026-09-02,Shipped,2499
C003,O1003,2026-09-03,Delivered,1599
C004,O1004,2026-09-04,Cancelled,699
C005,O1005,2026-09-05,Delivered,3299
C006,O1006,2026-09-06,Processing,2199
C007,O1007,2026-09-07,Delivered,1899
C008,O1008,2026-09-08,Returned,899
C009,O1009,2026-09-09,Delivered,3997
C010,O1010,2026-09-10,Shipped,1499


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.orders
USING DELTA
LOCATION 'abfss://silver@ecommerce1darsana.dfs.core.windows.net/orders';

In [0]:
orders_silver_check_df = spark.read \
    .format("delta") \
    .load("abfss://silver@ecommerce1darsana.dfs.core.windows.net/orders")

display(orders_silver_check_df)

customer_id,order_id,order_date,order_status,order_total
C001,O1001,2026-09-01,Delivered,2298
C002,O1002,2026-09-02,Shipped,2499
C003,O1003,2026-09-03,Delivered,1599
C004,O1004,2026-09-04,Cancelled,699
C005,O1005,2026-09-05,Delivered,3299
C006,O1006,2026-09-06,Processing,2199
C007,O1007,2026-09-07,Delivered,1899
C008,O1008,2026-09-08,Returned,899
C009,O1009,2026-09-09,Delivered,3997
C010,O1010,2026-09-10,Shipped,1499


In [0]:
silver_path = "abfss://silver@ecommerce1darsana.dfs.core.windows.net/orders"

orders_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("Orders Silver written successfully.")

Orders Silver written successfully.


In [0]:

from pyspark.sql.functions import col, coalesce, expr, to_date

orders_silver_df = orders_silver_df.withColumn(
    "order_date",
    to_date(
        coalesce(
            expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
            expr("try_to_timestamp(order_date, 'yyyy/MM/dd')")
        )
    )
)

orders_silver_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



In [0]:

print("Orders DataFrame schema:")
orders_clean_df.printSchema()

print("Sample order dates:")
orders_clean_df.select("order_id", "order_date").show(10, truncate=False)

Orders DataFrame schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)

Sample order dates:
+--------+----------+
|order_id|order_date|
+--------+----------+
|O1001   |2026-09-01|
|O1002   |2026-09-02|
|O1003   |2026-09-03|
|O1004   |2026-09-04|
|O1005   |2026-09-05|
|O1006   |2026-09-06|
|O1007   |2026-09-07|
|O1008   |2026-09-08|
|O1009   |2026-09-09|
|O1010   |2026-09-10|
+--------+----------+
only showing top 10 rows


In [0]:
display(
    orders_clean_df.alias("o")
    .join(
        spark.table("ecommerce.silver.customers").alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left_anti"
    )
)

order_id,customer_id,order_date,order_status,order_total
O1011,C999,2026-09-11,Processing,799


In [0]:
from pyspark.sql.functions import col, sum

print("=== ORDERS DATA QUALITY REPORT ===")

# Null values
print("\nNull values:")
orders_clean_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in orders_clean_df.columns
    ]
).show()

# Negative totals
negative_total_count = (
    orders_clean_df
    .filter(col("order_total") < 0)
    .count()
)
print("Negative order totals:", negative_total_count)

# Invalid dates
invalid_date_count = (
    orders_clean_df
    .filter(col("order_date").isNull())
    .count()
)
print("Invalid dates:", invalid_date_count)

# Invalid customer IDs
customers_df_silver = (
    spark.table("ecommerce.silver.customers")
    .select("customer_id")
)

invalid_customer_count = (
    orders_clean_df
    .join(
        customers_df_silver,
        "customer_id",
        "left_anti"
    )
    .count()
)

print("Invalid customer IDs:", invalid_customer_count)

# Order status distribution
print("\nOrder status values:")
orders_clean_df.groupBy("order_status").count().show()

=== ORDERS DATA QUALITY REPORT ===

Null values:
+--------+-----------+----------+------------+-----------+
|order_id|customer_id|order_date|order_status|order_total|
+--------+-----------+----------+------------+-----------+
|       0|          0|         0|           0|          0|
+--------+-----------+----------+------------+-----------+

Negative order totals: 0
Invalid dates: 0
Invalid customer IDs: 1

Order status values:
+------------+-----+
|order_status|count|
+------------+-----+
|    Returned|    1|
|     Shipped|    2|
|  Processing|    2|
|   Cancelled|    1|
|   Delivered|    7|
+------------+-----+



In [0]:
from pyspark.sql.functions import col

print("=== ORDERS DATA QUALITY REPORT ===")

# 1. Null values
print("\nNull values:")
orders_clean_df.select(
    *[
        col(c).isNull().cast("int").alias(c)
        for c in orders_clean_df.columns
    ]
).groupBy().sum().show()

# 2. Negative order totals
negative_total_count = (
    orders_clean_df
    .filter(col("order_total") < 0)
    .count()
)
print("Negative order totals:", negative_total_count)

# 3. Invalid/missing dates
invalid_date_count = (
    orders_clean_df
    .filter(col("order_date").isNull())
    .count()
)
print("Invalid dates:", invalid_date_count)

# 4. Customer IDs that don't exist
customers_df_silver = spark.table("ecommerce.silver.customers").select("customer_id")

invalid_customer_count = (
    orders_clean_df
    .join(
        customers_df_silver,
        "customer_id",
        "left_anti"
    )
    .count()
)
print("Invalid customer IDs:", invalid_customer_count)

# 5. Order status values
print("\nOrder status values:")
orders_clean_df.groupBy("order_status").count().show()

=== ORDERS DATA QUALITY REPORT ===

Null values:
+-------------+----------------+---------------+-----------------+----------------+
|sum(order_id)|sum(customer_id)|sum(order_date)|sum(order_status)|sum(order_total)|
+-------------+----------------+---------------+-----------------+----------------+
|            0|               0|              0|                0|               0|
+-------------+----------------+---------------+-----------------+----------------+

Negative order totals: 0
Invalid dates: 0
Invalid customer IDs: 1

Order status values:
+------------+-----+
|order_status|count|
+------------+-----+
|    Returned|    1|
|     Shipped|    2|
|  Processing|    2|
|   Cancelled|    1|
|   Delivered|    7|
+------------+-----+



In [0]:
from pyspark.sql.functions import col

orders_clean_df = orders_clean_df.withColumn(
    "order_total",
    col("order_total").cast("int")
)

orders_clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



In [0]:
invalid_date_count = (
    orders_clean_df
    .filter(col("order_date").isNull())
    .count()
)

print("Orders with invalid dates:", invalid_date_count)

Orders with invalid dates: 0


In [0]:
from pyspark.sql.functions import to_date, coalesce, col

orders_clean_df = orders_clean_df.withColumn(
    "order_date",
    coalesce(
        to_date(col("order_date"), "yyyy-MM-dd"),
        to_date(col("order_date"), "yyyy/MM/dd")
    )
)

orders_clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



In [0]:
display(
    orders_clean_df.alias("o")
    .join(
        spark.table("ecommerce.silver.customers").alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left_anti"
    )
)

order_id,customer_id,order_date,order_status,order_total
O1011,C999,2026-09-11,Processing,799


In [0]:
from pyspark.sql.functions import col

orders_alias = orders_clean_df.alias("o")
customers_alias = spark.table("ecommerce.silver.customers").alias("c")

invalid_customer_count = (
    orders_alias
    .join(
        customers_alias,
        col("o.customer_id") == col("c.customer_id"),
        "left_anti"
    )
    .count()
)

print("Orders with invalid customer IDs:", invalid_customer_count)

Orders with invalid customer IDs: 1


In [0]:

from pyspark.sql.functions import col

orders = orders_clean_df.alias("o")
customers = spark.table("ecommerce.silver.customers").alias("c")

invalid_customer_count = (
    orders
    .join(
        customers,
        col("o.customer_id") == col("c.customer_id"),
        "left_anti"
    )
    .count()
)

print("Orders with invalid customer IDs:", invalid_customer_count)

Orders with invalid customer IDs: 1


In [0]:
invalid_total_count = (
    orders_clean_df
    .filter(col("order_total") < 0)
    .count()
)

print("Orders with negative totals:", invalid_total_count)

Orders with negative totals: 0


In [0]:
from pyspark.sql.functions import to_date

orders_clean_df = orders_clean_df.withColumn(
    "order_date",
    to_date("order_date")
)

orders_clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum

orders_clean_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in orders_clean_df.columns
    ]
).show()

+--------+-----------+----------+------------+-----------+
|order_id|customer_id|order_date|order_status|order_total|
+--------+-----------+----------+------------+-----------+
|       0|          0|         0|           0|          0|
+--------+-----------+----------+------------+-----------+



In [0]:
from pyspark.sql.functions import col, sum

orders_clean_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in orders_clean_df.columns
    ]
).show()

+--------+-----------+----------+------------+-----------+
|order_id|customer_id|order_date|order_status|order_total|
+--------+-----------+----------+------------+-----------+
|       0|          0|         0|           0|          0|
+--------+-----------+----------+------------+-----------+



In [0]:

from pyspark.sql.functions import col, sum

customers_df = spark.table("ecommerce.silver.customers")

customers_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in customers_df.columns
    ]
).show()

+-----------+-------------+-----+-------+-----+-----------+
|customer_id|customer_name|email|country|state|signup_date|
+-----------+-------------+-----+-------+-----+-----------+
|          0|            0|    0|      0|    0|          0|
+-----------+-------------+-----+-------+-----+-----------+



In [0]:
%sql
SELECT * 
FROM ecommerce.silver.customers;

customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05


In [0]:
%sql DESCRIBE TABLE ecommerce.silver.customers;

col_name,data_type,comment
customer_id,string,null
customer_name,string,null
email,string,null
country,string,null
state,string,null
signup_date,date,null


In [0]:
silver_df = spark.read \
    .format("delta") \
    .load("abfss://silver@ecommerce1darsana.dfs.core.windows.net/customers")

display(silver_df)

customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05


In [0]:
customers_clean_df = customers_df.dropDuplicates(["customer_id"])

display(customers_clean_df)

customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05


In [0]:
display(
    customers_clean_df.filter(
        customers_clean_df.customer_name.isNull()
    )
)

customer_id,customer_name,email,country,state,signup_date


In [0]:
print("Total rows:", customers_clean_df.count())

print("Missing customer IDs:",
      customers_clean_df.filter(customers_clean_df.customer_id.isNull()).count())

print("Missing customer names:",
      customers_clean_df.filter(customers_clean_df.customer_name.isNull()).count())

Total rows: 11
Missing customer IDs: 0
Missing customer names: 0


In [0]:
silver_path = "abfss://silver@ecommerce1darsana.dfs.core.windows.net/customers"

customers_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
silver_path = "abfss://silver@ecommerce1darsana.dfs.core.windows.net/customers"

customers_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
from pyspark.sql.functions import coalesce, lit

customers_clean_df = customers_df.dropDuplicates(["customer_id"])

customers_clean_df = customers_clean_df.withColumn(
    "customer_name",
    coalesce(customers_clean_df.customer_name, lit("Unknown"))
)

display(customers_clean_df)

customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05


In [0]:
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/ecommerce/bronze/raw_files/customers.csv")

display(customers_df)

customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05


In [0]:
from pyspark.sql.functions import coalesce, lit

customers_clean_df = customers_clean_df.withColumn(
    "customer_name",
    coalesce(customers_clean_df.customer_name, lit("Unknown"))
)

display(customers_clean_df)

customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05


In [0]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [0]:
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/ecommerce/bronze/raw_files/customers.csv")

display(customers_df)

customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05


In [0]:
display(dbutils.fs.ls("/Volumes/ecommerce/bronze/raw_files/"))

path,name,size,modificationTime
dbfs:/Volumes/ecommerce/bronze/raw_files/customers.csv,customers.csv,845,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/order_items.csv,order_items.csv,429,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/orders.csv,orders.csv,566,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/payments.csv,payments.csv,620,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/products.csv,products.csv,538,1790838266000


In [0]:
display(dbutils.fs.ls("/Volumes/ecommerce/bronze/raw_files/"))

path,name,size,modificationTime
dbfs:/Volumes/ecommerce/bronze/raw_files/customers.csv,customers.csv,845,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/order_items.csv,order_items.csv,429,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/orders.csv,orders.csv,566,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/payments.csv,payments.csv,620,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/products.csv,products.csv,538,1790838266000


In [0]:
%sql
SHOW GRANTS ON EXTERNAL LOCATION ecommerce_bronze;

Principal,ActionType,ObjectType,ObjectKey
darsanaunni70@gmail.com,CREATE EXTERNAL VOLUME,EXTERNAL LOCATION,ecommerce_bronze


In [0]:
%sql
SHOW STORAGE CREDENTIALS;

name,comment
adb_ecommerce,null
ecommerce_adls_credential,null


In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

name,url,comment
adb_ecommerce,abfss://unity-catalog-storage@dbstorage6yl545zmyxuy4.dfs.core.windows.net/7405612586568834,null
ecommerce_bronze,abfss://bronze@ecommerce1darsana.dfs.core.windows.net/,null
ecommerce_gold,abfss://gold@ecommerce1darsana.dfs.core.windows.net/,null
ecommerce_silver,abfss://silver@ecommerce1darsana.dfs.core.windows.net/,null


In [0]:
%sql
SELECT * 
FROM read_files(
  '/Volumes/ecommerce/bronze/raw_files/customers.csv',
  format => 'csv',
  header => true
)
LIMIT 5;

customer_id,customer_name,email,country,state,signup_date,_rescued_data
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15,null
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18,null
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02,null
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10,null
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15,null


In [0]:
%sql
DESCRIBE VOLUME ecommerce.bronze.raw_files;

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
raw_files,ecommerce,bronze,darsanaunni70@gmail.com,abfss://bronze@ecommerce1darsana.dfs.core.windows.net/,EXTERNAL,null,VOLUME,VOLUME_EXTERNAL


In [0]:
%sql
DESCRIBE EXTERNAL LOCATION `ecommerce_bronze`;

name,url,credential_name,owner,created_by,created_at,comment
ecommerce_bronze,abfss://bronze@ecommerce1darsana.dfs.core.windows.net/,ecommerce_adls_credential,darsanaunni70@gmail.com,darsanaunni70@gmail.com,2026-10-02T10:43:25.927Z,null


In [0]:
%sql
DESCRIBE STORAGE CREDENTIAL `adb_ecommerce`;

name,owner,created_at,created_by,credential,comment
adb_ecommerce,_workspace_admins_adb_ecommerce_7405612586568834,2026-10-02T09:44:32.774Z,60bef642-ed8a-4aa8-b0a4-a04a541c7954,"AzureManagedIdentity:connector_id:/subscriptions/77c8cbc2-0aed-462f-928f-1a7d2911cb66/resourcegroups/databricks-rg-adb_ecommerce-lizaoiljwyc4y/providers/Microsoft.Databricks/accessConnectors/unity-catalog-access-connector,mi_id:System-Assigned MI,credential_id:7a2cba04-cc39-40a0-8506-1eea9ea402d3",null


In [0]:
display(dbutils.fs.ls("/Volumes/ecommerce/bronze/raw_files/"))

path,name,size,modificationTime
dbfs:/Volumes/ecommerce/bronze/raw_files/customers.csv,customers.csv,845,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/order_items.csv,order_items.csv,429,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/orders.csv,orders.csv,566,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/payments.csv,payments.csv,620,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/products.csv,products.csv,538,1790838266000


In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/ecommerce/bronze/raw_files/customers.csv")

display(df)

customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05


In [0]:

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/ecommerce/bronze/raw_files/customers.csv")

display(df)

customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05


In [0]:
%sql
SELECT * 
FROM csv.`/Volumes/ecommerce/bronze/raw_files/customers.csv`
LIMIT 5;

_c0,_c1,_c2,_c3,_c4,_c5
customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10


In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/ecommerce/bronze/raw_files/customers.csv")

display(df)

customer_id,customer_name,email,country,state,signup_date
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05


In [0]:
display(dbutils.fs.ls("/Volumes/ecommerce/bronze/raw_files/"))

path,name,size,modificationTime
dbfs:/Volumes/ecommerce/bronze/raw_files/customers.csv,customers.csv,845,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/order_items.csv,order_items.csv,429,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/orders.csv,orders.csv,566,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/payments.csv,payments.csv,620,1790838266000
dbfs:/Volumes/ecommerce/bronze/raw_files/products.csv,products.csv,538,1790838266000


In [0]:
%sql
GRANT READ VOLUME ON VOLUME ecommerce.bronze.raw_files TO `darsanaunni70@gmail.com`;

In [0]:
%sql
GRANT USE SCHEMA ON SCHEMA ecommerce.bronze TO `darsanaunni70@gmail.com`;

In [0]:
%sql
GRANT USE CATALOG ON CATALOG ecommerce TO `darsanaunni70@gmail.com`;

In [0]:
%sql
SHOW GRANTS ON SCHEMA ecommerce.bronze;

Principal,ActionType,ObjectType,ObjectKey
darsanaunni70@gmail.com,USE SCHEMA,SCHEMA,ecommerce.bronze


In [0]:
%sql
SHOW GRANTS ON CATALOG ecommerce;

Principal,ActionType,ObjectType,ObjectKey
darsanaunni70@gmail.com,USE CATALOG,CATALOG,ecommerce
account users,BROWSE,CATALOG,ecommerce


In [0]:
%sql
SHOW GRANTS ON VOLUME ecommerce.bronze.raw_files;

Principal,ActionType,ObjectType,ObjectKey
darsanaunni70@gmail.com,READ VOLUME,VOLUME,ecommerce.bronze.raw_files


In [0]:
%sql DESCRIBE VOLUME ecommerce.bronze.raw_files

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
raw_files,ecommerce,bronze,darsanaunni70@gmail.com,abfss://bronze@ecommerce1darsana.dfs.core.windows.net/,EXTERNAL,null,VOLUME,VOLUME_EXTERNAL
